# Lab 3.1 &mdash; Memory That Survives a Long Session

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 1 &middot; Module 3 &mdash; Memory, State &amp; the LangGraph Substrate**

### What you'll do
- Measure the turn at which buffer memory loses its standing instruction
- Implement summary compaction that keeps the constraint, not just the recent text
- Add episodic memory -- what happened last time, and whether that helps
- Decide deliberately what gets forgotten

> **How this lab works.** Fill every `BLANK`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, so your score never depends on a
> live endpoint. Cells marked **Run it for real** do call the sandbox model; if it is not
> reachable they print how to fix it instead of crashing.

> **Builds on Lab 1.2's `ShortTermMemory`.** There you kept the window bounded.
> Here you find out what bounding it cost you, and fix the part that mattered.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-3-01")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- graded cells still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 1 labs: payment exceptions on a small ledger.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

## Concept

Buffer memory grows linearly and the context window is fixed, so there is always a turn **N** at
which the earliest content falls out. Usually that content is the standing instruction, so the
agent quietly stops obeying it. Nothing errors.

Compaction bounds the window, but it is **lossy by construction**. The engineering question is not
whether to lose something &mdash; it is whether you chose what.

## Section 1 &mdash; Find the failure turn

Before fixing anything, measure it. Given a window budget and an average turn size, at what turn
does the standing instruction fall out of a pure buffer?

In [ ]:
WINDOW_TOKENS = 800                  # a deliberately small window, so the effect is visible
INSTRUCTION_TOKENS = 60              # the standing instruction, sent first
TURN_TOKENS = 40                     # an average turn

def failure_turn(window=WINDOW_TOKENS, instruction=INSTRUCTION_TOKENS, per_turn=TURN_TOKENS):
    """The first turn number at which the instruction no longer fits alongside the turns.

    Buffer memory sends: instruction + every turn so far. Once that exceeds the window,
    the oldest content -- the instruction -- is what gets dropped.
    """
    turn = 0
    while True:
        turn += 1
        used = instruction + turn * per_turn
        if BLANK:                    # TODO: has the buffer outgrown the window?
            return turn

In [ ]:
# --- Self-check: Section 1
check("the failure turn is computed, not guessed", lambda: failure_turn() == 19,
      "60 + 19*40 = 820 > 800, while 60 + 18*40 = 780 still fits")
check("a bigger window survives longer",
      lambda: failure_turn(window=4000) > failure_turn(window=800))
check("chattier turns fail sooner",
      lambda: failure_turn(per_turn=200) < failure_turn(per_turn=40))
check("the instruction's own size matters",
      lambda: failure_turn(instruction=600) < failure_turn(instruction=60))

for w in (800, 4000, 32000):
    try:
        print(f"  window {w:>6} tokens -> instruction drops out at turn {failure_turn(window=w)}")
    except NameError:
        print("(fill in failure_turn above)"); break

## Section 2 &mdash; Compaction that keeps what matters

Lab 1.2 kept the recent half and folded the rest into a summary. That bounds the window but can
still lose the standing instruction. Pin it instead.

In [ ]:
class Memory:
    """Recent turns verbatim, older ones summarised, and a pinned instruction that never ages out."""

    def __init__(self, instruction: str, max_turns: int = 6):
        self.instruction = instruction
        self.max_turns = max_turns
        self.turns: list[tuple[str, str]] = []
        self.summary = ""

    def add(self, role: str, text: str) -> None:
        self.turns.append((role, text))
        if len(self.turns) > self.max_turns:
            self.compact()

    def compact(self) -> None:
        keep = max(1, self.max_turns // 2)
        older, self.turns = self.turns[:-keep], self.turns[-keep:]
        self.summary = (self.summary + " " + " ".join(t for _, t in older)).strip()

    def render(self) -> list[tuple[str, str]]:
        """The messages to send. The instruction is pinned FIRST and always present."""
        out = BLANK                  # TODO: start with the pinned instruction as a system message
        if self.summary:
            out = out + [("system", "Earlier in this case: " + self.summary)]
        return out + list(self.turns)

In [ ]:
# --- Self-check: Section 2
INSTRUCTION = "Never propose releasing a payment that policy reserves for a human."

def _aged(n=40):
    m = Memory(INSTRUCTION, max_turns=6)
    for i in range(n):
        m.add("human" if i % 2 == 0 else "ai", f"turn {i}")
    return m

check("the window stays bounded after 40 turns", lambda: len(_aged().turns) <= 6)
check("the instruction is still present at turn 40",
      lambda: any(INSTRUCTION in t for _, t in _aged().render()),
      "pin it in render() -- it must not be subject to compaction")
check("the instruction comes first", lambda: _aged().render()[0][1] == INSTRUCTION)
check("the summary sits between the instruction and the recent turns",
      lambda: _aged().render()[1][0] == "system" and "Earlier" in _aged().render()[1][1])
check("a short conversation carries no summary",
      lambda: len(Memory(INSTRUCTION, 6).render()) == 1)

## Section 3 &mdash; What compaction threw away

Bounded is not the same as harmless. Measure the loss: which specific facts from early turns can no
longer be recovered from the rendered context?

In [ ]:
def recoverable(memory: Memory, fact: str) -> bool:
    """True when `fact` can still be found anywhere in what would be sent to the model."""
    return any(fact.lower() in text.lower() for _, text in memory.render())

def compaction_loss(facts: list[str], turns: int = 40) -> list[str]:
    """Run a session of `turns` turns, stating each fact early, and return the facts lost."""
    m = Memory(INSTRUCTION, max_turns=6)
    for f in facts:
        m.add("human", f)
    for i in range(turns):
        m.add("ai", f"working, step {i}")
    return BLANK                     # TODO: the facts that are no longer recoverable

In [ ]:
# --- Self-check: Section 3
FACTS = ["The case reference is PMT-1005.",
         "The client contact is the Frankfurt desk.",
         "Do not contact the counterparty directly."]

check("concatenating summaries keeps early facts recoverable",
      lambda: compaction_loss(FACTS) == [],
      "this compaction folds text rather than discarding it -- so nothing is lost YET")
check("the instruction survives regardless",
      lambda: recoverable(_aged(), "reserves for a human"))

# ...but a summariser that REWRITES rather than concatenates does lose things:
class LossyMemory(Memory):
    def compact(self):
        keep = max(1, self.max_turns // 2)
        older, self.turns = self.turns[:-keep], self.turns[-keep:]
        self.summary = f"[{len(older)} earlier turns summarised]"   # a real summariser, abbreviating

def _lossy():
    m = LossyMemory(INSTRUCTION, max_turns=6)
    for f in FACTS:
        m.add("human", f)
    for i in range(40):
        m.add("ai", f"step {i}")
    return m

check("a rewriting summariser DOES lose the early facts",
      lambda: not recoverable(_lossy(), "Frankfurt"),
      "this is the real behaviour of an LLM summariser, and the reason to pin what matters")
check("...but the pinned instruction still survives it",
      lambda: recoverable(_lossy(), "reserves for a human"),
      "pinning is what makes compaction safe to use")

## Run it for real

Ask the model to compact, which is what a production summariser does &mdash; then check whether your
facts survived it.

In [ ]:
if llm_ready():
    try:
        transcript = " ".join([
            "The case reference is PMT-1005.",
            "The client contact is the Frankfurt desk.",
            "Do not contact the counterparty directly.",
        ] + [f"Analyst checked step {i} and found nothing unusual." for i in range(20)])

        summary = ask(
            "Summarise this case transcript in at most two sentences for an operations handover.\n\n"
            + transcript)
        print("SUMMARY:\n  " + summary.strip().replace("\n", "\n  "))
        print("\nsurvived compaction?")
        for f in ("PMT-1005", "Frankfurt", "counterparty"):
            print(f"  {f:16} {'yes' if f.lower() in summary.lower() else 'NO -- lost'}")
    except NameError:
        print("(fill in the blanks above, then re-run this cell)")

### Read it

Whatever the model dropped, it dropped **silently and plausibly** &mdash; the summary reads fine. That is
the whole risk: compaction failures are invisible at the point they happen and only surface later,
as an agent that has stopped honouring something it was told.

Two defences, in order: **pin** what must never be lost, and **test** what your summariser keeps
against a list of facts you care about. This lab is that test.

In [ ]:
score()

## Your turn

1. Add an `episodic` list to `Memory` holding one line per past case, and include the three most
   recent in `render()`. Then ask: when would recalling a past case make the agent *worse*?
2. Pinning costs tokens on every single turn. Work out the break-even: how long must a session be
   before pinning a 60-token instruction is cheaper than the failure it prevents?